In [1]:
import numpy as np
import time
import platform
from pathlib import Path
from skimage import io
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from PIL import Image
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, AveragePooling2D, Dense, Flatten, InputLayer
from tqdm import tqdm
import os
import gc


In [2]:
# if torch.cuda.is_available():
#     device = 'cuda'
# elif torch.mps.is_available():
#     device = 'mps'
# else:
#     'cpu'
    
# print(device)

devices = tf.config.list_physical_devices()
print("\nDevices: ", devices)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
  details = tf.config.experimental.get_device_details(gpus[0])
  print("GPU details: ", details)


Devices:  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU details:  {'device_name': 'METAL'}


In [3]:
import kagglehub

path = kagglehub.competition_download('2-computer-vision-2026-b-sc-aidams-final-proj')

print("Path to competition files:", path)

/Users/iamgeorgerieh/Documents/computer-vision-class/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 248M/248M [00:06<00:00, 39.4MB/s] 

Extracting files...


Path to competition files: /Users/iamgeorgerieh/.cache/kagglehub/competitions/2-computer-vision-2026-b-sc-aidams-final-proj


In [ ]:
# def load_data(folder_path:str):
#     path_to_images = Path(folder_path) 
#     ids = []
#     dataset = []
#     for img_file in sorted(path_to_images.iterdir()):
#         id_img = "".join(img_file.name.split(".")[:-1])
#         ids.append(id_img)
#         im = Image.open(img_file)
#         # if im.format == 'PNG' and im.mode != 'RGBA': 
#             # encountered multiple errors look like UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
#         img_file = f'{img_file}2.png'
#         im = im.convert("RGBA").save(img_file)
        
#         img = io.imread(img_file, as_gray=True)
#         dataset.append(img)
#     dataset_as_array = np.array(dataset, dtype="object")
#     return dataset_as_array, ids

In [4]:
def load_data(folder_path):
    pbar = tqdm(total=len([name for name in os.listdir(folder_path) if os.path.isfile(name)]))
    path_to_images = Path(folder_path)
    ids = []
    dataset = []
    for img_file in sorted(path_to_images.iterdir()):
        id_img = "".join(img_file.name.split(".")[:-1])
        ids.append(id_img)
        im = Image.open(img_file)
        if im.format == 'PNG' and im.mode != 'RGBA':
            img_file = f'{img_file}2.png'
            im.convert("RGBA").save(img_file)
        img = io.imread(img_file)
        gray_weighted = img[:,:,0] * 0.299 + 0.587 * img[:,:,1] + 0.114 * img[:,:,2]
        gray_weighted = tf.image.resize([gray_weighted], [72, 72])
        gray_weighted = tf.image.random_brightness(gray_weighted, max_delta=0.1) 
        gray_weighted = tf.image.random_contrast(gray_weighted, lower=0.9, upper=1.1)
        dataset.append(gray_weighted)
        pbar.update(1)
    
    dataset_as_array = np.array(dataset, dtype='object')
    pbar.close()
    return dataset_as_array, ids


In [4]:
def load_data(folder_path):
    path_to_images = Path(folder_path)
    image_files = [f for f in sorted(path_to_images.iterdir()) if f.is_file()]
    
    ids = []
    dataset = []
    
    pbar = tqdm(total=len(image_files))
    
    for img_file in image_files:
        id_img = img_file.stem
        ids.append(id_img)
        
        with Image.open(img_file) as im:
            if im.format == 'PNG' and im.mode != 'RGBA':
                converted_path = img_file.parent / f"{img_file.stem}_conv.png"
                im.convert("RGBA").save(converted_path)
                img_file = converted_path

        img = io.imread(img_file)
        
        gray_weighted = img[:, :, 0] * 0.299 + 0.587 * img[:, :, 1] + 0.114 * img[:, :, 2]
        
        gray_weighted = np.expand_dims(gray_weighted, axis=-1)
        
        resized_tensor = tf.image.resize(gray_weighted, [72, 72])
        resized_arr = resized_tensor.numpy().astype(np.float32) / 255.0
        
        dataset.append(resized_arr)
        
        del img, gray_weighted, resized_tensor
        pbar.update(1)
    
    pbar.close()
    

    dataset_as_array = np.array(dataset, dtype=np.float32)
    
    gc.collect() #asked ai for this part, because I was doing transformation inside the loop and without garbage collectio n
    
    return dataset_as_array, ids

In [5]:
X_train, X_train_ids = load_data(path+"/train")
print("Shape of train dataset", X_train.shape)

X_test, X_test_ids = load_data(path+"/test")
print("Shape of test dataset", X_test.shape)
y_train = pd.read_csv(path+"/train_labels.csv")
print("Shape of predictions for train dataset", y_train.shape)

  0%|          | 0/9879 [00:00<?, ?it/s]2026-09-17 20:55:42.508240: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-09-17 20:55:42.508334: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-09-17 20:55:42.508354: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-09-17 20:55:42.509170: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-17 20:55:42.511318: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
100%|██████████| 9879/9879 [00:21<00:00, 469.25it/s]


Shape of train dataset (9879, 72, 72, 1)


100%|██████████| 9879/9879 [00:20<00:00, 474.93it/s]


Shape of test dataset (9879, 72, 72, 1)
Shape of predictions for train dataset (9879, 2)


In [6]:
X_train_norm = X_train.astype("float32") / 255.0
X_test_norm = X_test.astype("float32") / 255.0

X_train_norm = np.expand_dims(X_train_norm, axis=-1)
X_test_norm = np.expand_dims(X_test_norm, axis=-1)

print("Train:", X_train_norm.shape)
print("Test:", X_test_norm.shape)

Train: (9879, 72, 72, 1, 1)
Test: (9879, 72, 72, 1, 1)


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomBrightness(factor=0.1),  
    tf.keras.layers.RandomContrast(factor=0.1),    
])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(72, 72, 1)),
    data_augmentation, 
    
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation='softmax')
])